# Construction Materials NLP — Data Exploration & Cleaning

**Project:** `demo_genai` · **Classes:** Concrete, Metals, Wood, Insulation, Glass

This notebook explores noisy synthetic invoice descriptions, creates a reproducible cleaning pipeline, and exports a model-ready dataset. **Run from the notebook directory or the repository root.**

> **Important:** Synthetic examples often reuse material names and templates. Random train/test splits can overstate real-world performance. Keep a separate, independently labeled test set for meaningful evaluation.

## 1. Setup

Install once in your project environment:

```powershell
python -m pip install pandas numpy matplotlib scikit-learn jupyter ipykernel
```

In VS Code, select the Python interpreter from `demo_genai/.venv`.

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_colwidth", 120)

# Works whether Jupyter starts in repository root or notebooks/.
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Cannot find project root with data/ directory. Open demo_genai in VS Code.")

DATA_PATH = ROOT / "data" / "sample" / "construction_materials_large.csv"
OUTPUT_PATH = ROOT / "data" / "processed" / "construction_materials_clean.csv"
print("Input:", DATA_PATH)
print("Output:", OUTPUT_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH}. Run scripts/generate_material_dataset.py first.")

## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH, dtype="string", keep_default_na=True)
required = {"description", "category"}
missing_columns = required - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
print("Columns:", list(df.columns))
display(df.head(10))
display(df.dtypes.to_frame("dtype"))

## 3. Missing values, empty text, and duplicates

In [ ]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_non_null": df.nunique(dropna=True),
})
display(quality)

blank_desc = df["description"].fillna("").str.strip().eq("")
blank_category = df["category"].fillna("").str.strip().eq("")
exact_duplicates = df.duplicated(subset=["description", "category"]).sum()
conflicting_labels = (
    df.dropna(subset=["description", "category"])
      .groupby("description")["category"].nunique().gt(1).sum()
)
print("Blank descriptions:", int(blank_desc.sum()))
print("Blank categories:", int(blank_category.sum()))
print("Exact duplicate description+category:", int(exact_duplicates))
print("Descriptions assigned multiple categories:", int(conflicting_labels))

## 4. Category distribution

In [ ]:
counts = df["category"].value_counts(dropna=False)
display(counts.rename_axis("category").reset_index(name="count"))
ax = counts.plot(kind="bar", figsize=(9, 4), title="Material category distribution")
ax.set_xlabel("Category")
ax.set_ylabel("Number of descriptions")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Text length and noisy descriptions

In [ ]:
eda = df.copy()
eda["char_count"] = eda["description"].fillna("").str.len()
eda["word_count"] = eda["description"].fillna("").str.split().str.len()
display(eda[["char_count", "word_count"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eda["char_count"].hist(bins=35, ax=axes[0])
axes[0].set_title("Description length (characters)")
axes[0].set_xlabel("Characters")
eda["word_count"].hist(bins=25, ax=axes[1])
axes[1].set_title("Description length (words)")
axes[1].set_xlabel("Words")
plt.tight_layout()
plt.show()

if "noise_type" in df.columns:
    # Multiple noise tags may be separated by '|'.
    tags = df["noise_type"].fillna("none").str.split("|").explode().value_counts()
    display(tags.rename_axis("noise_type").reset_index(name="count"))
    display(df.loc[df["noise_type"].fillna("none").ne("none"),
                   ["description", "category", "noise_type"]].head(15))

## 6. Text normalization

For the baseline TF-IDF model, use **conservative cleaning**:

- Normalize Unicode and case, including Dutch text.
- Standardize decimal punctuation and common units (`m³` → `m3`, `100 mm` → `100mm`).
- Expand a small list of known abbreviations.
- Keep digits, slashes, plus signs, periods, and hyphens because material grades and dimensions matter.
- Remove synthetic invoice metadata (PO numbers, order references, prices, and project names) when identifiable.

**Do not automatically remove stopwords, apply stemming, or guess spelling corrections.** Those operations can damage material names or technical specifications. The spelling mistakes generated by the script are deliberately left in place for robust modeling.

In [ ]:
ABBREVIATIONS = {
    "conc.": "concrete",
    "reinfrc.": "reinforced",
    "galv.": "galvanized",
    "tmbr": "timber",
    "plywd": "plywood",
    "insul.": "insulation",
    "isol.pl.": "isolatieplaat",
    "dbl glazed": "double glazed",
    "veil.glas": "veiligheidsglas",
    "gew.": "gewapend",
    "verz.": "verzinkte",
    "alu": "aluminium",
    "rvs": "rvs",  # Keep common Dutch stainless-steel abbreviation
    "ss": "ss",    # Context-dependent; don't blindly expand
}

# Longest first, word boundaries prevent partial replacements inside other words.
ABBREV_RE = re.compile(
    r"(?<!\w)(?:" + "|".join(re.escape(k) for k in sorted(ABBREVIATIONS, key=len, reverse=True)) + r")(?!\w)",
    flags=re.IGNORECASE,
)

def clean_description(value: object) -> str:
    """Conservatively normalize an invoice description without destroying material specs."""
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).lower().strip()
    text = text.replace("³", "3").replace("²", "2")
    text = ABBREV_RE.sub(lambda m: ABBREVIATIONS[m.group(0).lower()], text)

    # Remove clearly synthetic reference and price fields (not arbitrary numbers).
    text = re.sub(r"\bpo[- ]?\d{5,}\b", " ", text)
    text = re.sub(r"\bref\s+\d{5}\b", " ", text)
    text = re.sub(r"\beur\s+\d+(?:[.,]\d+)?\b", " ", text)

    # Normalize dimensions and units, preserving technical identifiers.
    text = re.sub(r"(\d)\s+(mm|cm|m2|m3|kg)\b", r"\1\2", text)
    text = re.sub(r"\s*[;,]\s*", " ", text)
    text = re.sub(r"[^\w\s./+%x-]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

examples = [
    "BETONMORTEL C30/37 25 m³ PO-123456",
    "galv. steel HEA200, 12 pcs EUR 450",
    "PIR isol.pl. 100 mm Rd 4.5",
    "HR++ dubbel glas 4-15-4 12 m²",
]
for item in examples:
    print(f"Before: {item}\nAfter:  {clean_description(item)}\n")

## 7. Apply cleaning and inspect before/after

In [ ]:
cleaned = df.copy()
cleaned["description_clean"] = cleaned["description"].map(clean_description).astype("string")
cleaned["category"] = cleaned["category"].str.strip()

# Remove records without usable input or target.
cleaned = cleaned.loc[
    cleaned["description_clean"].str.len().gt(0).fillna(False)
    & cleaned["category"].notna()
    & cleaned["category"].ne("")
].copy()

print("Rows after removing empty inputs/labels:", len(cleaned))
display(cleaned[["description", "description_clean", "category"]].sample(
    min(15, len(cleaned)), random_state=42
))

## 8. Remove duplicates and detect label conflicts

Identical cleaned text with different labels is ambiguous for a supervised classifier. We **exclude those conflicting groups** and review them separately. Identical cleaned text with the same label is deduplicated.

For a real dataset, review conflicting labels manually instead of deleting them automatically.

In [ ]:
label_counts = cleaned.groupby("description_clean")["category"].nunique()
conflict_texts = label_counts[label_counts > 1].index
conflicts = cleaned[cleaned["description_clean"].isin(conflict_texts)].copy()
print("Conflicting cleaned descriptions:", len(conflict_texts))
if len(conflicts):
    display(conflicts[["description", "description_clean", "category"]].head(20))

before = len(cleaned)
cleaned = cleaned.loc[~cleaned["description_clean"].isin(conflict_texts)].copy()
cleaned = cleaned.drop_duplicates(subset=["description_clean", "category"])
print(f"Removed {before - len(cleaned):,} conflicting/duplicate rows")
print(f"Final unique rows: {len(cleaned):,}")
display(cleaned["category"].value_counts().rename("count").to_frame())

## 9. Check cleaned text length and most frequent terms

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

if cleaned.empty:
    raise ValueError("No rows remain after cleaning.")

vectorizer = CountVectorizer(max_features=30, token_pattern=r"(?u)\b\w[\w/+.-]*\b")
counts_matrix = vectorizer.fit_transform(cleaned["description_clean"])
term_counts = np.asarray(counts_matrix.sum(axis=0)).ravel()
top_terms = pd.DataFrame({
    "term": vectorizer.get_feature_names_out(),
    "count": term_counts,
}).sort_values("count", ascending=False)
display(top_terms.head(20))

## 10. Export the cleaned dataset

The exported CSV contains `description`, `description_clean`, and `category` (plus `noise_type` if available). Use **`description_clean` as the input feature** for the first model. Keep `description` for error analysis and traceability.

Your `.gitignore` may exclude `data/processed/`; this is fine if you want the generator and cleaning notebook to be the reproducible source of truth.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cols = ["description", "description_clean", "category"]
if "noise_type" in cleaned.columns:
    cols.append("noise_type")
cleaned[cols].to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved {len(cleaned):,} rows to {OUTPUT_PATH}")
display(cleaned[cols].head())

## 11. Next steps: training and evaluation

1. Load `data/processed/construction_materials_clean.csv`.
2. Use `description_clean` as `X` and `category` as `y`.
3. Build a `Pipeline(TfidfVectorizer(...), LogisticRegression(...))`.
4. Compare results on **clean**, **noisy**, and **previously unseen material names**.
5. Keep the final test data independent of training and preprocessing decisions.

**Leakage caution:** If you later fit a vectorizer, feature selector, or learned text normalizer, fit it on the **training split only**. This notebook's cleaning is deterministic and does not learn vocabulary or statistics from the dataset. Avoid allowing near-duplicate generated templates across training and test splits.